In [ ]:
print("hello")
from google.colab import files
uploaded = files.upload()

hello


Saving source_es_segmented.txt to source_es_segmented.txt


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving styleguide_es_en.md to styleguide_es_en.md


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving target_deepl_en.txt to target_deepl_en (1).txt
Saving target_gemini_en.txt to target_gemini_en (1).txt
Saving target_gtranslate_en.txt to target_gtranslate_en (1).txt


In [ ]:
import os
for f in sorted(os.listdir()):
    if f.endswith(('.md', '.txt')):
        print(f)

source_es_segmented.txt
styleguide_es_en.md
target_deepl_en.txt
target_gemini_en.txt
target_gtranslate_en.txt


In [ ]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))
print("key loaded")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


key loaded


In [ ]:
!pip install -q google-genai pydantic
styleguide = open('styleguide_es_en.md', encoding='utf-8').read()
source     = open('source_es_segmented.txt', encoding='utf-8').read()
target     = open('target_gemini_en.txt', encoding='utf-8').read()

print("styleguide:", len(styleguide), "chars")
print("source:", len(source), "chars")
print("target:", len(target), "chars")


styleguide: 14883 chars
source: 4966 chars
target: 4872 chars


In [ ]:
from google.colab import userdata
from google import genai
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
print("client ready")

client ready


In [ ]:
import re
def split_segments(text):
    parts = re.split(r'(\[S\d+\])', text)
    return {parts[i].strip('[]'): parts[i+1].strip() for i in range(1, len(parts), 2)}
src = split_segments(source)
tgt = split_segments(target)
print("segments:", len(src), len(tgt))

segments: 26 26


In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
print("client ready")

client ready


In [ ]:
print(len(styleguide), len(src), "client" in dir())

14883 26 True


In [ ]:
import re
valid_ids = sorted(set(re.findall(r'\b(?:VOICE|ADDR|TERM|DNT|MEAS|FIX|FORMAT|PACE|TRAP|ACC)-\d+\b', styleguide)))
print(len(valid_ids), "rule IDs:")
print(valid_ids)

41 rule IDs:
['ACC-01', 'ACC-02', 'ACC-03', 'ACC-04', 'ADDR-01', 'ADDR-02', 'DNT-01', 'DNT-02', 'DNT-03', 'FIX-01', 'FIX-02', 'FORMAT-01', 'FORMAT-02', 'FORMAT-03', 'FORMAT-04', 'MEAS-01', 'MEAS-02', 'MEAS-03', 'MEAS-04', 'MEAS-05', 'MEAS-06', 'MEAS-07', 'PACE-01', 'PACE-02', 'PACE-03', 'PACE-04', 'PACE-05', 'TERM-01', 'TERM-02', 'TERM-03', 'TERM-04', 'TERM-05', 'TRAP-01', 'TRAP-02', 'TRAP-03', 'TRAP-04', 'TRAP-05', 'VOICE-01', 'VOICE-02', 'VOICE-03', 'VOICE-04']


In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with the single word: working"
)
print(response.text)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}

In [ ]:
notes = """
# Translation Styleguide Judge — progress notes

## Setup (cells above)
1. Uploaded 5 files: styleguide, source, 3 target translations (deepl, google, gemini)
2. Loaded them into variables: styleguide, source, target
3. Connected to Gemini API (client)
4. Split source & target into 26 segments each (src, tgt)
5. Extracted 41 valid rule IDs from the styleguide (valid_ids)

## What we found yesterday (judge run on ~10 segments, Gemini)
- Good at surface rules (spelling, punctuation, added words)
- Blind to pace/structure rules (needs comparing whole sentences)
- Sometimes misapplies rule IDs
- Note: "PACE-04" was NOT invented, it's a real renumbered rule

## Doing now
- Adding Pydantic validation:
  - enforces the shape of each violation
  - flags any rule_id NOT in the 41 valid IDs (catches invented rules)
  - does NOT catch real-but-wrong IDs (harder problem)

## Next
- Re-run judge with validation
- Later: split into specialist agents (mechanical / accuracy / pace)
"""

In [ ]:
from pydantic import BaseModel, field_validator
from typing import Literal

class Violation(BaseModel):
    segment: str
    rule_id: str
    severity: Literal["critical", "major", "minor"]
    source_span: str
    target_span: str
    note: str

    @field_validator("rule_id")
    @classmethod
    def check_rule_id(cls, v):
        if v not in valid_ids:
            raise ValueError(f"unknown rule_id: {v}")
        return v

print("Violation model ready")

Violation model ready


In [ ]:
## Pydantic validation — what it does

Pydantic is a format checker, not a knowledge graph. It does NOT understand the rules or their meaning.

It only checks:
- each violation has the 6 required fields
- severity is one of: critical / major / minor
- rule_id is one of the 41 valid IDs (catches INVENTED rules)

It does NOT check whether a valid rule was applied correctly (a real-but-wrong ID still passes). Format only, not meaning.

In [ ]:
import time
from pydantic import ValidationError

def judge_segment(seg_id, src_text, tgt_text):
    prompt = f"""You are a translation reviewer. Compare the Spanish source to the English target for segment {seg_id}, using the styleguide.

Return a JSON list of violations. Each has: segment, rule_id, severity (critical/major/minor), source_span, target_span, note.
Use ONLY rule_ids that appear in the styleguide. If no violations, return [].

STYLEGUIDE:
{styleguide}

SOURCE [{seg_id}]:
{src_text}

TARGET [{seg_id}]:
{tgt_text}
"""
    wait = 5
    for _ in range(6):
        try:
            r = client.models.generate_content(
                model="gemini-3.5-flash",
                contents=prompt,
                config={"response_mime_type": "application/json"}
            )
            return r.text
        except Exception:
            time.sleep(wait); wait *= 2
    return None

print("judge function ready")

judge function ready


In [ ]:
## Why clean JSON + Pydantic

Yesterday the judge returned raw text (sometimes wrapped in ```json fences).
Result: we couldn't process it automatically — we had to read every cell by hand to check the rules.

Fix, two parts:
- response_mime_type="application/json" → Gemini returns clean data, no fences
- Pydantic → parses each violation, flags invented rule IDs automatically

Payoff: the manual eyeball check becomes automatic — the system checks all 26 segments and flags problems itself.

In [ ]:
import json

raw = judge_segment("S04", src["S04"], tgt["S04"])
print("RAW OUTPUT:\n", raw)

print("\nVALIDATION:")
for v in json.loads(raw):
    try:
        Violation(**v)
        print("  OK  ", v["rule_id"])
    except Exception as e:
        print("  FAIL", v.get("rule_id"), "-", e)

RAW OUTPUT:
 [
  {
    "segment": "S04",
    "rule_id": "TERM-01",
    "severity": "major",
    "source_span": "Tinajas",
    "target_span": "Large jars",
    "note": "The glossary specifies 'storage jar' for 'tinaja'. 'Large jars' is a violation."
  },
  {
    "segment": "S04",
    "rule_id": "TERM-01",
    "severity": "major",
    "source_span": "ollas",
    "target_span": "pots",
    "note": "The glossary specifies 'cooking pot' for 'olla'. 'pots' is a violation."
  },
  {
    "segment": "S04",
    "rule_id": "ACC-03",
    "severity": "minor",
    "source_span": "el batir",
    "target_span": "clattering",
    "note": "The styleguide states that the sound of unglazed ceramic is hollow, and rules that 'knocking' or 'clacking' must be used for 'el batir de unas piezas'."
  }
]


VALIDATION:
  OK   TERM-01
  OK   TERM-01
  OK   ACC-03


In [ ]:
import json, time

all_violations = []
flagged = []
errors = []

for sid in src:
    if sid not in tgt:
        continue
    print("judging", sid)
    raw = judge_segment(sid, src[sid], tgt[sid])
    if raw is None:
        errors.append(sid); print("   ERROR"); continue
    try:
        for v in json.loads(raw):
            try:
                Violation(**v)
                all_violations.append(v)
            except Exception as e:
                flagged.append({"seg": sid, "data": v, "why": str(e)})
    except Exception as e:
        errors.append(sid); print("   parse fail")
    time.sleep(4)

print("\n=== DONE ===")
print("valid violations:", len(all_violations))
print("flagged (bad id/shape):", len(flagged))
print("errors (quota/parse):", len(errors))

judging S01
judging S02
judging S03
judging S04
judging S05
judging S06
judging S07
judging S08
judging S09
judging S10
judging S11
judging S12
judging S13
judging S14
judging S15
judging S16
judging S17
judging S18
judging S19
judging S20
judging S21
judging S22
   ERROR
judging S23
judging S24
judging S25
   ERROR
judging S26
   ERROR

=== DONE ===
valid violations: 41
flagged (bad id/shape): 0
errors (quota/parse): 3


In [ ]:
import json
with open('judge_validated_gemini.json', 'w') as f:
    json.dump({"violations": all_violations, "flagged": flagged, "errors": errors}, f, indent=2, ensure_ascii=False)
from google.colab import files
files.download('judge_validated_gemini.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
validation_results_notes = """
## Run with Pydantic validation results

Ran the judge on 26 segments (Gemini). 23 judged, 3 errors (S22, S25, S26 quota).

Results:
- 41 valid violations (correct shape, real rule IDs)
- 0 flagged (no invented rule IDs)
- 3 errors

## What Pydantic did / didn't do
- It DETECTS, it does not CORRECT. It compares each result against the spec (6 fields, valid severity, rule_id in the list of 41).
- It catches INVENTED rule IDs only. We found zero across both runs.
- Note: yesterday's "PACE-04 hallucination" was a misread PACE-04 is a real rule.

## The judge's real weaknesses (Pydantic CANNOT catch these):
- misapplied valid IDs (e.g. MEAS-05 on the wrong line)
- missed violations (e.g. PACE-03 structural rule on S04)
These need human judgment or a smarter check, not format validation.
"""

In [ ]:
print(len(all_violations), "client" in dir(), len(src))

41 True 26


In [ ]:
"""
NEXT: is pace-blindness the MODEL or the PROMPT?

Finding: judge misses pace rules (PACE-01, PACE-03).
Test: re-run one pace-heavy segment with a prompt that explicitly
asks it to compare sentence lengths and rhythm.
- catches it now -> was the PROMPT
- still misses -> the MODEL can't do relational judgment
Build agents only if the focused prompt helps.
"""

"\nNEXT: is pace-blindness the MODEL or the PROMPT?\n\nFinding: judge misses pace rules (PACE-01, PACE-03).\nTest: re-run one pace-heavy segment with a prompt that explicitly\nasks it to compare sentence lengths and rhythm.\n- catches it now -> was the PROMPT\n- still misses -> the MODEL can't do relational judgment\nBuild agents only if the focused prompt helps.\n"

In [ ]:
pace_prompt = f"""You are a translation reviewer focused ONLY on pace and rhythm.

Compare the Spanish source to the English target. Do this specifically:
1. Count the sentences in the source and in the target.
2. Compare their average length.
3. The Spanish source uses deliberately SHORT sentences (short by Spanish norms) to create a calm, unhurried tone.
4. Check: does the English preserve that? Or are the sentences longer / merged / smoothed, losing the calm?

Relevant rules:
- PACE-01: match the EFFECT (calm pace), not the punctuation marks
- PACE-03: never merge sentences the source keeps separate

Return a JSON list of violations (segment, rule_id, severity, source_span, target_span, note). If none, return [].

SOURCE [S01]:
{src['S01']}

TARGET [S01]:
{tgt['S01']}
"""

r = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=pace_prompt,
    config={"response_mime_type": "application/json"}
)
print(r.text)

[
  {
    "segment": "S01",
    "rule_id": "PACE-03",
    "severity": "major",
    "source_span": "Se tarda mucho en dominar la técnica, en entender cómo se comporta el barro.",
    "target_span": "It takes a long time to master the technique and understand how the clay behaves.",
    "note": "The target merged two clauses that were presented as distinct, measured thoughts, creating a faster, more flowing pace than the source."
  },
  {
    "segment": "S01",
    "rule_id": "PACE-03",
    "severity": "major",
    "source_span": "Incluso en distinguir cuándo una pieza está bien hecha y ver cómo, a base de práctica, las líneas se van afinando.",
    "target_span": "Even just learning to tell when a piece is well-made and watching how, with enough practice, the lines become finer and cleaner.",
    "note": "By merging the 'distinguishing' and 'watching' actions into one long, complex sentence, the target loses the fragmented, unhurried rhythm of the original."
  }
]


In [ ]:
with open('pace_test_S01.json', 'w') as f:
    f.write(r.text)
from google.colab import files
files.download('pace_test_S01.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
PACE TEST RESULT — question answered

Q: Is pace-blindness the MODEL or the PROMPT?
A: The PROMPT.

- General judge MISSED pace on S01.
- Focused pace-prompt CAUGHT it: 2x PACE-03, correctly flagging merged clauses
  and lost rhythm.

Corrected finding:
  NOT "the model can't perceive structure"
  BUT "a single generalist judge misses structural rules; a focused,
       decomposed prompt recovers them. Failure was task design, not
       model capability."

This justifies the specialist-agent design: split the judge by rule type,
give each a focused prompt.

Caveat: one segment, one run. Directionally clear, small sample.
"""

'\nPACE TEST RESULT — question answered\n\nQ: Is pace-blindness the MODEL or the PROMPT?\nA: The PROMPT.\n\n- General judge MISSED pace on S01.\n- Focused pace-prompt CAUGHT it: 2x PACE-03, correctly flagging merged clauses\n  and lost rhythm.\n\nCorrected finding:\n  NOT "the model can\'t perceive structure"\n  BUT "a single generalist judge misses structural rules; a focused,\n       decomposed prompt recovers them. Failure was task design, not\n       model capability."\n\nThis justifies the specialist-agent design: split the judge by rule type,\ngive each a focused prompt.\n\nCaveat: one segment, one run. Directionally clear, small sample.\n'

In [ ]:
for m in client.models.list():
    if 'flash' in m.name.lower():
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/gemini-3.1-flash-tts-preview
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Reply with the single word: working"
)
print(response.text)

working


In [ ]:
rules_prompt = f"""You are a translation reviewer. Compare the Spanish source to the English target for ONE segment, using the styleguide.

Return ONLY a JSON list of violations. Each violation:
{{"segment": "...", "rule_id": "...", "severity": "critical|major|minor", "source_span": "...", "target_span": "...", "note": "..."}}

If no violations, return [].
Use only rule_ids that exist in the styleguide.

STYLEGUIDE:
{styleguide}

SOURCE SEGMENT:
[S04] En casa de mi madre había piezas de cerámica muy grandes. Tinajas, ollas, que venían de la aldea de mi abuela, del interior de Galicia, en el norte de España.

TARGET SEGMENT:
In my mother's house there were very large ceramic pieces. Jars and pots, which came from my grandmother's village, in the Galician countryside, in northern Spain.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=rules_prompt
)
print(response.text)

[
  {
    "segment": "In my mother's house there were very large ceramic pieces. Jars and pots, which came from my grandmother's village, in the Galician countryside, in northern Spain.",
    "rule_id": "ACC-04",
    "severity": "major",
    "source_span": "del interior de Galicia",
    "target_span": "in the Galician countryside",
    "note": "The styleguide explicitly bans translating 'del interior de Galicia' as 'the Galician countryside' because countryside can be coastal. It must be translated as 'inland Galicia' or 'the interior of Galicia'."
  },
  {
    "segment": "In my mother's house there were very large ceramic pieces. Jars and pots, which came from my grandmother's village, in the Galician countryside, in northern Spain.",
    "rule_id": "TERM-01",
    "severity": "minor",
    "source_span": "Tinajas",
    "target_span": "Jars",
    "note": "The glossary requires 'tinaja' to be translated as 'storage jar'."
  },
  {
    "segment": "In my mother's house there were very larg

In [ ]:
import re

def split_segments(text):
    # split on [S01], [S02]... keeping the tag
    parts = re.split(r'(\[S\d+\])', text)
    segs = {}
    for i in range(1, len(parts), 2):
        tag = parts[i].strip('[]')
        segs[tag] = parts[i+1].strip()
    return segs

src = split_segments(source)
tgt = split_segments(target)   # target = gemini, loaded earlier

print("source segments:", len(src))
print("target segments:", len(tgt))
print("S04 source:", src.get("S04"))

source segments: 26
target segments: 26
S04 source: En casa de mi madre había piezas de cerámica muy grandes. Tinajas, ollas, que venían de la aldea de mi abuela, del interior de Galicia, en el norte de España. Mi madre contaba que de niña prestaban los carros de bueyes para llevar las piezas desde Gundivós, donde se hacían, y que se oía el batir de unas piezas contra otras por el camino.


In [ ]:
import json, time

def judge_segment(seg_id, src_text, tgt_text):
    prompt = f"""You are a translation reviewer. Compare the Spanish source to the English target for ONE segment, using the styleguide.

Return ONLY a JSON list of violations. Each: {{"segment":"{seg_id}","rule_id":"...","severity":"critical|major|minor","source_span":"...","target_span":"...","note":"..."}}
If none, return []. Use only rule_ids from the styleguide.

STYLEGUIDE:
{styleguide}

SOURCE [{seg_id}]:
{src_text}

TARGET [{seg_id}]:
{tgt_text}
"""
    r = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
    return r.text

all_results = {}
for sid in src:
    if sid in tgt:
        print("judging", sid, "...")
        all_results[sid] = judge_segment(sid, src[sid], tgt[sid])
        time.sleep(1)

print("done —", len(all_results), "segments judged")

judging S01 ...
judging S02 ...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
import json, time

def judge_segment(seg_id, src_text, tgt_text):
    prompt = f"""You are a translation reviewer. Compare the Spanish source to the English target for ONE segment, using the styleguide.

Return ONLY a JSON list of violations. Each: {{"segment":"{seg_id}","rule_id":"...","severity":"critical|major|minor","source_span":"...","target_span":"...","note":"..."}}
If none, return []. Use only rule_ids from the styleguide.

STYLEGUIDE:
{styleguide}

SOURCE [{seg_id}]:
{src_text}

TARGET [{seg_id}]:
{tgt_text}
"""
    r = client.models.generate_content(model="gemini-flash-lite-latest", contents=prompt)
    return r.text

all_results = {}
for sid in src:
    if sid in tgt:
        print("judging", sid, "...")
        all_results[sid] = judge_segment(sid, src[sid], tgt[sid])
        time.sleep(1)

print("done —", len(all_results), "segments judged")

judging S01 ...
judging S02 ...
judging S03 ...
judging S04 ...
judging S05 ...
judging S06 ...
judging S07 ...
judging S08 ...
judging S09 ...
judging S10 ...
judging S11 ...
judging S12 ...
judging S13 ...
judging S14 ...
judging S15 ...
judging S16 ...
judging S17 ...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.1-flash-lite\nPlease retry in 2.587270005s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.1-f

In [ ]:
import json, time

def judge_segment(seg_id, src_text, tgt_text):
    prompt = f"""You are a translation reviewer. Compare the Spanish source to the English target for ONE segment, using the styleguide.

Return ONLY a JSON list of violations. Each: {{"segment":"{seg_id}","rule_id":"...","severity":"critical|major|minor","source_span":"...","target_span":"...","note":"..."}}
If none, return []. Use only rule_ids from the styleguide.

STYLEGUIDE:
{styleguide}

SOURCE [{seg_id}]:
{src_text}

TARGET [{seg_id}]:
{tgt_text}
"""
    wait = 5
    for attempt in range(6):
        try:
            r = client.models.generate_content(model="gemini-2.5-flash-lite", contents=prompt)
            return r.text
        except Exception as e:
            print("   busy, waiting", wait, "s")
            time.sleep(wait)
            wait = wait * 2   # 5,10,20,40...
    return "ERROR"

try:
    all_results
except NameError:
    all_results = {}

for sid in src:
    if sid in tgt and sid not in all_results:
        print("judging", sid)
        all_results[sid] = judge_segment(sid, src[sid], tgt[sid])
        time.sleep(5)

print("done —", len(all_results), "of", len(src), "segments")

judging S17
   busy, waiting 5 s
   busy, waiting 10 s
   busy, waiting 20 s
   busy, waiting 40 s
   busy, waiting 80 s
   busy, waiting 160 s
judging S18
   busy, waiting 5 s
   busy, waiting 10 s
   busy, waiting 20 s
   busy, waiting 40 s
   busy, waiting 80 s
   busy, waiting 160 s
judging S19
   busy, waiting 5 s
   busy, waiting 10 s
   busy, waiting 20 s
   busy, waiting 40 s
   busy, waiting 80 s


KeyboardInterrupt: 

In [ ]:
import json
with open('judge_results_gemini.json', 'w') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

from google.colab import files
files.download('judge_results_gemini.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
for sid, text in all_results.items():
    print("="*40, sid)
    print(text)

======================================== S01
[
  {
    "segment": "S01",
    "rule_id": "TERM-05",
    "severity": "major",
    "source_span": "aprendiendo cerámica en torno",
    "target_span": "learning wheel-thrown pottery",
    "note": "The styleguide explicitly prescribes 'learning to throw on the wheel' for this exact sentence instead of using the compound adjective 'wheel-thrown'."
  },
  {
    "segment": "S01",
    "rule_id": "ACC-01",
    "severity": "major",
    "source_span": "las líneas se van afinando",
    "target_span": "the lines become finer and cleaner",
    "note": "The styleguide explicitly forbids translating 'las líneas se van afinando' as 'finer and cleaner', as it adds an extra idea not present in the source. It should be 'the lines become more refined'."
  }
]
======================================== S02
```json
[
  {
    "segment": "S02",
    "rule_id": "ACC-02",
    "severity": "minor",
    "source_span": "frente a la playa",
    "target_span": "right across 

In [ ]:
"""
CORRECTION — the rule-ID problem, stated precisely

Two different errors, don't confuse them:

1. INVENTING IDs (a rule that doesn't exist) — did NOT happen. Pydantic found zero.
   Note: yesterday's "PACE-04 hallucination" was a misread — PACE-04 is a real rule.

2. MISALIGNING IDs (a real rule ID cited for the WRONG violation) — DID happen.
   Example S08: flagged the weight but cited MEAS-05, which is actually the
   separator rule. Valid ID, wrong match.

Why it matters:
- Pydantic only checks "is this ID on the list?" A misaligned ID is on the list,
  so it PASSES. Validation caught nothing because this error is wrong-but-valid.
- This class needs human judgment or a semantic check — not format validation.

So: the judge misaligns real IDs; it does not fabricate them.
"""

In [1]:
print(len(all_violations), "client" in dir(), len(src), len(valid_ids))

NameError: name 'all_violations' is not defined

In [2]:
!pip install -q google-genai pydantic

In [3]:
from google.colab import files
uploaded = files.upload()

Saving styleguide_es_en.md to styleguide_es_en.md


In [4]:
from google.colab import files
uploaded = files.upload()

Saving source_es_segmented.txt to source_es_segmented.txt


In [5]:
from google.colab import files
uploaded = files.upload()

Saving target_deepl_en.txt to target_deepl_en.txt
Saving target_gemini_en.txt to target_gemini_en.txt
Saving target_gtranslate_en.txt to target_gtranslate_en.txt


In [6]:
styleguide = open('styleguide_es_en.md', encoding='utf-8').read()
source     = open('source_es_segmented.txt', encoding='utf-8').read()
target     = open('target_gemini_en.txt', encoding='utf-8').read()
print("loaded")

loaded


In [7]:
from google.colab import userdata
from google import genai
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
print("client ready")

client ready


In [8]:
import re
def split_segments(text):
    parts = re.split(r'(\[S\d+\])', text)
    return {parts[i].strip('[]'): parts[i+1].strip() for i in range(1, len(parts), 2)}
src = split_segments(source)
tgt = split_segments(target)
valid_ids = sorted(set(re.findall(r'\b(?:VOICE|ADDR|TERM|DNT|MEAS|FIX|FORMAT|PACE|TRAP|ACC)-\d+\b', styleguide)))
print("segments:", len(src), "| rule ids:", len(valid_ids))

segments: 26 | rule ids: 41


In [9]:
"""
=== SESSION 3 — plan ===

Setup restored (files, client, 26 segments, 41 rule IDs, Pydantic model). Done

STEP 1 — Test the pace finding across MORE segments
  Yesterday: focused pace prompt caught PACE errors on S01 that the
  generalist missed. But n=1.
  Now: run the pace prompt on the 5 prose segments (S01-S05) to see if
  it holds. Product blocks (S07+) are specs, not pace-relevant.

STEP 2 — Build 3 judges and compare
  - generalist  (all rules at once)   = yesterday's judge
  - terminology (TERM rules only)
  - pace        (PACE rules only)
  Question: do the 2 specialists together catch what the generalist missed?

"""

"\n=== SESSION 3 — plan ===\n\nSetup restored (files, client, 26 segments, 41 rule IDs, Pydantic model).\n\nSTEP 1 — Test the pace finding across MORE segments\n  Yesterday: focused pace prompt caught PACE errors on S01 that the\n  generalist missed. But n=1.\n  Now: run the pace prompt on the 5 prose segments (S01-S05) to see if\n  it holds. Product blocks (S07+) are specs, not pace-relevant.\n\nSTEP 2 — Build 3 judges and compare\n  - generalist  (all rules at once)   = yesterday's judge\n  - terminology (TERM rules only)\n  - pace        (PACE rules only)\n  Question: do the 2 specialists together catch what the generalist missed?\n\n"

In [10]:
from pydantic import BaseModel, field_validator
from typing import Literal

class Violation(BaseModel):
    segment: str
    rule_id: str
    severity: Literal["critical", "major", "minor"]
    source_span: str
    target_span: str
    note: str
    @field_validator("rule_id")
    @classmethod
    def check(cls, v):
        if v not in valid_ids:
            raise ValueError(f"unknown rule_id: {v}")
        return v
print("model ready")

model ready


In [11]:
import json, time

pace_results = {}
for sid in ["S01","S02","S03","S04","S05"]:
    prompt = f"""You are a translation reviewer focused ONLY on pace and rhythm.

Compare the Spanish source to the English target:
1. Count sentences in source vs target.
2. Compare average length.
3. The Spanish uses deliberately SHORT sentences (short by Spanish norms) for a calm, unhurried tone.
4. Does the English preserve that, or are sentences longer / merged / smoothed?

Rules:
- PACE-01: match the EFFECT (calm pace), not the marks
- PACE-03: never merge sentences the source keeps separate

Return a JSON list of violations (segment, rule_id, severity, source_span, target_span, note). If none, [].

SOURCE [{sid}]:
{src[sid]}

TARGET [{sid}]:
{tgt[sid]}
"""
    print("pace-judging", sid)
    r = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )
    pace_results[sid] = r.text
    time.sleep(5)

print("\ndone")
for sid, out in pace_results.items():
    print("="*30, sid)
    print(out)


pace-judging S01
pace-judging S02
pace-judging S03
pace-judging S04
pace-judging S05

done
============================== S01
[
  {
    "segment": "S01",
    "rule_id": "PACE-03",
    "severity": "high",
    "source_span": "Se tarda mucho en dominar la técnica, en entender cómo se comporta el barro.",
    "target_span": "It takes a long time to master the technique and understand how the clay behaves.",
    "note": "The source uses two distinct clauses separated by a comma to maintain a staccato rhythm; the English translation uses 'and' to merge these into a single, faster-flowing compound predicate."
  },
  {
    "segment": "S01",
    "rule_id": "PACE-03",
    "severity": "medium",
    "source_span": "Incluso en distinguir cuándo una pieza está bien hecha y ver cómo, a base de práctica, las líneas se van afinando.",
    "target_span": "Even just learning to tell when a piece is well-made and watching how, with enough practice, the lines become finer and cleaner.",
    "note": "The so

In [12]:
"""
STEP 1 RESULT — pace finding holds (n=5)

Ran the pace-focused prompt on S01-S05.
It caught PACE-03 / PACE-01 violations on ALL FIVE segments — merged
clauses, lengthened rhythm — the same errors the generalist judge missed.

So the earlier conclusion is confirmed and stronger:
  the pace-blindness was PROMPT DESIGN, not a model limit.
  A focused prompt recovers structural rules the generalist overlooks.
  (n=5 now, not n=1.)

NOTE — severity mismatch:
  The model returned "high"/"medium", but our Pydantic spec only allows
  critical/major/minor. Those will FAIL validation.
  Useful example of Pydantic catching output that ignores the spec.
  we change the prompt to make it more specific and match the error classification in the pydantic model, the severity scale
"""

'\nSTEP 1 RESULT — pace finding holds (n=5)\n\nRan the pace-focused prompt on S01-S05.\nIt caught PACE-03 / PACE-01 violations on ALL FIVE segments — merged\nclauses, lengthened rhythm — the same errors the generalist judge missed.\n\nSo the earlier conclusion is confirmed and stronger:\n  the pace-blindness was PROMPT DESIGN, not a model limit.\n  A focused prompt recovers structural rules the generalist overlooks.\n  (n=5 now, not n=1.)\n\nNOTE — severity mismatch:\n  The model returned "high"/"medium", but our Pydantic spec only allows\n  critical/major/minor. Those will FAIL validation.\n  Useful example of Pydantic catching output that ignores the spec.\n  we change the prompt to make it more specific and match the error classification in the pydantic model, the severity scale\n'

In [13]:
import json, time

pace_results = {}
for sid in ["S01","S02","S03","S04","S05"]:
    prompt = f"""You are a translation reviewer focused ONLY on pace and rhythm.

Compare the Spanish source to the English target:
1. Count sentences in source vs target.
2. Compare average length.
3. The Spanish uses deliberately SHORT sentences (short by Spanish norms) for a calm, unhurried tone.
4. Does the English preserve that, or are sentences longer / merged / smoothed?

Rules:
- PACE-01: match the EFFECT (calm pace), not the marks
- PACE-03: never merge sentences the source keeps separate

severity must be exactly one of: critical, major, minor

Return a JSON list of violations (segment, rule_id, severity, source_span, target_span, note). If none, [].

SOURCE [{sid}]:
{src[sid]}

TARGET [{sid}]:
{tgt[sid]}
"""
    print("pace-judging", sid)
    r = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )
    pace_results[sid] = r.text
    time.sleep(5)

print("done")

pace-judging S01
pace-judging S02
pace-judging S03
pace-judging S04
pace-judging S05
done


In [14]:
import json
valid, failed = [], []
for sid, raw in pace_results.items():
    for v in json.loads(raw):
        try:
            Violation(**v)
            valid.append(v)
        except Exception as e:
            failed.append((sid, v.get("rule_id"), v.get("severity"), str(e)[:50]))

print("valid:", len(valid))
print("failed:", len(failed))
for f in failed:
    print("  ", f)

valid: 5
failed: 0


In [15]:
"""
KEY DISTINCTION — format vs judgment

Pydantic validation (valid: 5, failed: 0) confirms FORMAT only:
  - right fields, allowed severity words, real rule IDs.
It does NOT confirm the judgment is correct:
  - whether "major" is the right severity
  - whether PACE-03 is the right rule
  - whether the span is genuinely a violation
That still needs a human.

So: a violation can PASS validation and still be a WRONG call.
Format can be automated. Correctness cannot. The human stays.

(Reading the 5 pace catches, they do look correct — but that's ME
judging, not Pydantic.)
"""

'\nKEY DISTINCTION — format vs judgment\n\nPydantic validation (valid: 5, failed: 0) confirms FORMAT only:\n  - right fields, allowed severity words, real rule IDs.\nIt does NOT confirm the judgment is correct:\n  - whether "major" is the right severity\n  - whether PACE-03 is the right rule\n  - whether the span is genuinely a violation\nThat still needs a human.\n\nSo: a violation can PASS validation and still be a WRONG call.\nFormat can be automated. Correctness cannot. The human stays.\n\n(Reading the 5 pace catches, they do look correct — but that\'s ME\njudging, not Pydantic.)\n'

In [21]:
"""
---
=== STEP 2 -- three judges compared ===

Build ONE flexible judge() function, give it three different focuses:
  GENERALIST  -- check ALL rules at once (= the original judge)
  TERMINOLOGY -- only TERM/glossary rules
  PACE        -- only PACE/rhythm rules

Run all three on the 5 prose segments (S01-S05).

QUESTION: do TERMINOLOGY + PACE together catch what the GENERALIST misses?
  Prediction (from Step 1): yes -- the generalist overlooks pace, the
  pace specialist catches it. Splitting by rule type should improve coverage.

This tests the core claim: the earlier 'blindness' was task design,
not model capability. Decomposition = better coverage.

---
WHY SPLIT INTO 3 JUDGES -- the hypothesis

Claim: focused prompts catch more than one general prompt.

WHY it should work:
  Not magic in the number 3. A focused prompt gives the model ONE job,
  so it does not spread attention thin across 41 rules and drop the hard,
  relational ones (like pace).
  The generalist was not INCAPABLE of pace -- it was OVERLOADED.
  Narrowing the task recovers the capability.

We are TESTING this, not asserting it:
  Step 1 showed the pace judge catches pace (n=5).
  Step 2 checks the comparison: does the generalist still miss them,
  and do the specialists together cover more?

---
"""

"\n---\n=== STEP 2 -- three judges compared ===\n\nBuild ONE flexible judge() function, give it three different focuses:\n  GENERALIST  -- check ALL rules at once (= the original judge)\n  TERMINOLOGY -- only TERM/glossary rules\n  PACE        -- only PACE/rhythm rules\n\nRun all three on the 5 prose segments (S01-S05).\n\nQUESTION: do TERMINOLOGY + PACE together catch what the GENERALIST misses?\n  Prediction (from Step 1): yes -- the generalist overlooks pace, the\n  pace specialist catches it. Splitting by rule type should improve coverage.\n\nThis tests the core claim: the earlier 'blindness' was task design,\nnot model capability. Decomposition = better coverage.\n\n---\nWHY SPLIT INTO 3 JUDGES -- the hypothesis\n\nClaim: focused prompts catch more than one general prompt.\n\nWHY it should work:\n  Not magic in the number 3. A focused prompt gives the model ONE job,\n  so it does not spread attention thin across 41 rules and drop the hard,\n  relational ones (like pace).\n  The ge

In [17]:
def judge(sid, focus_instructions):
    prompt = f"""You are a translation reviewer.
{focus_instructions}

severity must be exactly one of: critical, major, minor
Return a JSON list of violations (segment, rule_id, severity, source_span, target_span, note). If none, [].
Use only rule_ids from this styleguide:
{styleguide}

SOURCE [{sid}]:
{src[sid]}

TARGET [{sid}]:
{tgt[sid]}
"""
    r = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )
    return r.text

GENERALIST = "Check the translation against ALL styleguide rules."
TERMINOLOGY = "Focus ONLY on terminology (TERM rules) and the glossary. Ignore pace and formatting."
PACE = "Focus ONLY on pace and rhythm (PACE rules). Check if sentences are merged or lengthened, losing the calm tone."

print("judges ready")

judges ready


In [22]:
import json, time

comparison = {}
for sid in ["S01","S02","S03","S04","S05"]:
    comparison[sid] = {}
    for name, focus in [("generalist", GENERALIST), ("terminology", TERMINOLOGY), ("pace", PACE)]:
        raw = judge(sid, focus)
        try:
            ids = [v["rule_id"] for v in json.loads(raw)]
        except:
            ids = ["PARSE_FAIL"]
        comparison[sid][name] = ids
        time.sleep(4)
    print(sid, "done")

print("\n=== rule IDs caught by each judge ===")
for sid in comparison:
    print("\n", sid)
    for name in ["generalist","terminology","pace"]:
        print(f"  {name:12}", comparison[sid][name])

S01 done
S02 done
S03 done
S04 done
S05 done

=== rule IDs caught by each judge ===

 S01
  generalist   ['VOICE-01', 'ACC-01', 'TERM-05']
  terminology  ['ACC-01', 'VOICE-01', 'VOICE-02']
  pace         ['PACE-03', 'ACC-01', 'VOICE-01']

 S02
  generalist   ['ACC-02', 'PACE-02', 'VOICE-02']
  terminology  ['ACC-02', 'TERM-01', 'PACE-02', 'TRAP-04']
  pace         ['ACC-02', 'PACE-03']

 S03
  generalist   ['VOICE-01', 'ACC-01', 'FORMAT-04', 'PACE-04']
  terminology  ['VOICE-02', 'VOICE-01', 'VOICE-01', 'VOICE-01', 'FORMAT-04']
  pace         ['PACE-03', 'VOICE-01', 'VOICE-01']

 S04
  generalist   ['VOICE-01', 'ACC-01', 'TERM-01']
  terminology  ['TERM-01', 'ACC-03']
  pace         ['PACE-03', 'PACE-03']

 S05
  generalist   ['PACE-02', 'FORMAT-01', 'TRAP-03']
  terminology  ['FORMAT-01', 'PACE-02', 'TERM-01', 'TRAP-03']
  pace         ['PACE-02', 'PACE-03', 'PACE-03']


In [23]:
"""
STEP 2 RESULT — messier than expected

- PACE judge: caught PACE-03 on all 5 segments; generalist caught 0. Real win.
- TERMINOLOGY judge: did WORSE than generalist on terminology (missed TERM-01
  where generalist caught it). Unexpected — terminology is mechanical, should be easy.

PROBLEM: every run is a SINGLE sample. Gemini isn't deterministic.
The terminology "failure" may just be run-to-run VARIANCE, not a real effect.
We've been concluding from n=1 runs. Need to test variance before trusting any of this.

NEXT: run the same judge on the same segment several times → is the result stable or noisy?
"""

'\nSTEP 2 RESULT — messier than expected\n\n- PACE judge: caught PACE-03 on all 5 segments; generalist caught 0. Real win.\n- TERMINOLOGY judge: did WORSE than generalist on terminology (missed TERM-01\n  where generalist caught it). Unexpected — terminology is mechanical, should be easy.\n\nPROBLEM: every run is a SINGLE sample. Gemini isn\'t deterministic.\nThe terminology "failure" may just be run-to-run VARIANCE, not a real effect.\nWe\'ve been concluding from n=1 runs. Need to test variance before trusting any of this.\n\nNEXT: run the same judge on the same segment several times → is the result stable or noisy?\n'

In [24]:
import json, time

print("terminology judge on S04, 5 runs:")
for i in range(5):
    raw = judge("S04", TERMINOLOGY)
    try:
        ids = [v["rule_id"] for v in json.loads(raw)]
    except:
        ids = ["PARSE_FAIL"]
    print(f"  run {i+1}:", ids)
    time.sleep(4)

terminology judge on S04, 5 runs:
  run 1: ['TERM-01', 'TERM-01', 'VOICE-01', 'ACC-03']
  run 2: ['TERM-01', 'TERM-01']
  run 3: ['TERM-01', 'VOICE-04', 'ACC-03']
  run 4: ['TERM-01', 'TERM-01', 'ACC-03']
  run 5: ['TERM-01', 'ACC-03', 'VOICE-01']


In [25]:
"""
STEP 2 — CORRECTED CONCLUSION (after variance test)

Ran terminology judge on S04 five times:
  - TERM-01 (clear glossary rule): caught ALL 5 runs → STABLE
  - VOICE / ACC (soft, judgment rules): flickered run to run → NOISY

So my earlier "terminology judge is worse" was WRONG — it came from a
single unlucky run (S01). Variance, not a real weakness. Terminology IS
reliable, as expected (mechanical rule).

REAL FINDING:
  - Clear/mechanical rules → consistent across runs (trust single runs)
  - Soft/interpretive rules → noisy (need multiple runs to trust)
  - Single-run results MISLEAD you on soft rules.

PACE result stands: PACE-03 caught on 5 DIFFERENT segments (5/5).
  Stronger than the terminology scare (1 miss on 1 segment).
  Limitation noted: one run per segment, but 5 different texts = sturdy.

BIG LESSON: don't conclude from n=1. Clear rules are stable; soft rules
need repeated runs. This applies to the whole experiment.
"""

'\nSTEP 2 — CORRECTED CONCLUSION (after variance test)\n\nRan terminology judge on S04 five times:\n  - TERM-01 (clear glossary rule): caught ALL 5 runs → STABLE\n  - VOICE / ACC (soft, judgment rules): flickered run to run → NOISY\n\nSo my earlier "terminology judge is worse" was WRONG — it came from a\nsingle unlucky run (S01). Variance, not a real weakness. Terminology IS\nreliable, as expected (mechanical rule).\n\nREAL FINDING:\n  - Clear/mechanical rules → consistent across runs (trust single runs)\n  - Soft/interpretive rules → noisy (need multiple runs to trust)\n  - Single-run results MISLEAD you on soft rules.\n\nPACE result stands: PACE-03 caught on 5 DIFFERENT segments (5/5).\n  Stronger than the terminology scare (1 miss on 1 segment).\n  Limitation noted: one run per segment, but 5 different texts = sturdy.\n\nBIG LESSON: don\'t conclude from n=1. Clear rules are stable; soft rules\nneed repeated runs. This applies to the whole experiment.\n'